In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Run once in your environment:
!pip install sentence-transformers torch openpyxl

In [ ]:
import itertools
import numpy as np
import pandas as pd
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

In [ ]:
# Read the file and split by |
with open('Repetitions-EN-MISTRAL.txt', 'r', encoding='utf-8-sig') as f:
    content = f.read()

# Split all snippets by the | delimiter
snippets = content.split('|')

# Remove any empty strings (in case of trailing |)
snippets = [s.strip() for s in snippets if s.strip()]

# Create 12 arrays of 10 strings each
arrays = [snippets[i:i+10] for i in range(0, 120, 10)]

# Verify
print(f"Number of arrays: {len(arrays)}")  # Should be 12
print(f"Size of each array: {[len(arr) for arr in arrays]}")  # Should be [10, 10, ..., 10]

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# Cosine Similarity (https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)
# ════════════════════════════════════════════════════════════════════════════════

# Symmetric: matrix[i][j] == matrix[j][i]
# Only compute upper triangle, then mirror it

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

all_texts = [s for arr in arrays for s in arr]  # 120 strings total
all_embeddings = model.encode(all_texts, convert_to_tensor=True)  # Shape: [120, embedding_dim]
embeddings_arrays = [all_embeddings[i:i+10] for i in range(0, 120, 10)]

labels = [f"{i+1}" for i in range(10)]

all_matrices = []
for z in range(12):
    mat = np.eye(10)  # diagonal = 1.0
    for i in range(10):
        for j in range(i + 1, 10):
            score = F.cosine_similarity(
                embeddings_arrays[z][i].unsqueeze(0),
                embeddings_arrays[z][j].unsqueeze(0)
            ).item()
            mat[i][j] = score
            mat[j][i] = score
    all_matrices.append(pd.DataFrame(mat, index=labels, columns=labels))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# EXPORT — Save all matrices to a single Excel file (one sheet per metric)
# ════════════════════════════════════════════════════════════════════════════════
output_path = "similarity_results.xlsx"

matrix_titles = [
    "1-ICE-EN",
    "2-ICE-EN",
    "3-ICE-EN",
    "4-ICE-EN",
    "1-IRN-EN",
    "2-IRN-EN",
    "3-IRN-EN",
    "4-IRN-EN",
    "1-ASK-EN",
    "2-ASK-EN",
    "3-ASK-EN",
    "4-ASK-EN",
]

BLOCK_ROWS = 13   # 1 title + 1 header + 10 data + 1 blank gap at bottom
BLOCK_COLS = 13   # 1 index + 10 data + 2 blank gap columns

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # Write a placeholder to create the sheet
    pd.DataFrame().to_excel(writer, sheet_name='SemScore', index=False)
    ws = writer.sheets['SemScore']

    for z, df_mat in enumerate(all_matrices):
        grid_row = z // 2   # 0..5  (which of the 6 vertical blocks)
        grid_col = z % 2    # 0 or 1 (left or right column)

        start_row = grid_row * BLOCK_ROWS       # Excel row offset (0-indexed for pandas)
        start_col = grid_col * BLOCK_COLS       # Excel col offset (0-indexed for pandas)

        # Write title one row above the matrix
        ws.cell(
            row=start_row + 1,          # openpyxl is 1-indexed
            column=start_col + 2,       # +2: skip index column, start at first data col
            value=matrix_titles[z]
        )

        # Write the matrix (startrow/startcol are 0-indexed in pandas)
        df_mat.round(4).to_excel(
            writer,
            sheet_name='SemScore',
            startrow=start_row + 1,     # +1 to leave room for the title row
            startcol=start_col
        )

print(f"Results saved to {output_path}")